# LLM 보고서 프롬프트 실험 (v12)

고정된 24건 입력에 여러 프롬프트를 적용해 문체와 사실 정확성을 비교한다.

- **DB를 사용하지 않는다.** fixture CSV만 읽는다
- production DTO·클라이언트·파서·검증기를 그대로 import한다
- **검증에 걸린 응답도 버리지 않는다.** 사유만 기록하고 원문을 보존한다
- 검증 실패로 자동 재생성하지 않는다. 첫 응답 품질을 비교하기 위해서다

`validator_ok`와 "사람이 보기에 좋은 보고서"는 다른 개념이다. 둘이
엇갈리는 사례를 찾는 것이 이 노트북의 목적이다.


In [ ]:
# Cell 1 — 경로와 import
import json
import sys

from dataclasses import dataclass
from datetime import date as date_type
from pathlib import Path
from typing import Callable

import pandas as pd

BASE_DIR = Path.cwd().parent

if not (BASE_DIR / "pilos").exists():
    BASE_DIR = Path.cwd()

if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

# production 구현을 그대로 재사용한다. 실험용으로 다시 만들지 않는다.
from pilos.analysis.llm_report import (
    build_report_messages,
    classify_supply_state,
    validate_market_commentary_response,
)
from pilos.collection.llm_report_client import (
    REPORT_MAX_TOKENS,
    REPORT_TEMPERATURE,
    LlmReportClientSettings,
    LlmReportTransportError,
    OpenAICompatibleLlmReportClient,
    parse_market_commentary_response,
)
from pilos.dto.llm_report_dto import (
    PROMPT_VERSION,
    LlmSignalEvidence,
    ReportGenerationRequest,
)

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)

print(f"BASE_DIR={BASE_DIR}")
print(f"production prompt_version={PROMPT_VERSION}")


## Cell 2 — 실험 설정

여기 값만 바꿔서 사용한다. 아래 셀의 내부 코드는 수정하지 않는다.


In [ ]:
# Cell 2 — 실험 설정
EXPERIMENT_VERSION = "v12"
N_CASES = 24
FIXTURE_SEED = 20260808

# 실제 API를 호출할지 결정한다. 기본값은 호출하지 않는다.
ENABLE_LLM_CALL = False

# 같은 case_id + experiment_id가 이미 있으면 다시 호출할지 결정한다.
OVERWRITE_EXISTING = False

BASELINE = "A_baseline"
FRIENDLY = "B_friendly_interpreter"
MOBILE = "C_mobile_briefing"
FREE = "D_free_numeric"

RUN_VARIANTS = [BASELINE, FRIENDLY, MOBILE, FREE]

# None이면 fixture 전체를 실행한다. [485, 490]처럼 지정하면 그 case만 실행.
RUN_CASE_IDS = None

# provider·model·timeout은 .env에서 읽는다. 키를 노트북에 적지 않는다.
settings = LlmReportClientSettings.from_env()

print(f"provider={settings.provider}")
print(f"model={settings.model}")
print(f"output_mode={settings.output_mode}")
print(f"temperature={REPORT_TEMPERATURE}  max_tokens={REPORT_MAX_TOKENS}")


In [ ]:
# Cell 3 — 경로 정의
REVIEW_DIR = BASE_DIR / "data" / "review"
EXPERIMENT_DIR = BASE_DIR / "data" / "llm_report_experiment" / EXPERIMENT_VERSION
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_REVIEW_CSV = REVIEW_DIR / "llm_report_review_v11.csv"
SOURCE_EVIDENCE_CSV = REVIEW_DIR / "llm_report_signal_evidence_v11.csv"

CASES_CSV = EXPERIMENT_DIR / "cases_v11.csv"
RUNS_CSV = EXPERIMENT_DIR / "experiment_runs.csv"
COMPARISON_CSV = EXPERIMENT_DIR / "experiment_comparison.csv"
MANUAL_REVIEW_CSV = EXPERIMENT_DIR / "manual_review.csv"

print(f"실험 산출물 경로: {EXPERIMENT_DIR}")
print(f"fixture: {CASES_CSV.name} (존재={CASES_CSV.exists()})")


## Cell 4 — fixture 생성 또는 로드

`cases_v11.csv`가 있으면 그대로 읽는다. 없을 때만 v11 결과에서 24건을
선정하고 저장한다. **한 번 만들어진 뒤에는 다시 sampling하지 않는다.**
그래야 A/B/C/D와 이후 실험이 완전히 같은 입력을 쓴다.

단순 `head(24)`나 순수 random이 아니라 수급 방향·신호 등급·전일 변화·
평균 관계가 고루 섞이도록 고르고, v11에서 문제가 됐던 성격도 의도적으로
포함한다.


In [ ]:
# Cell 4 — fixture 생성 또는 로드
CHANGE_BUCKETS = ("큰 상승", "작은 상승", "변화 없음", "작은 하락", "큰 하락")
AVERAGE_BUCKETS = ("평균 위", "평균 근처", "평균 아래")
LEVELS = ("매우 낮음", "낮음", "보통", "높음", "매우 높음")


def classify_change(value: int) -> str:
    if value >= 15:
        return "큰 상승"
    if value >= 1:
        return "작은 상승"
    if value == 0:
        return "변화 없음"
    if value >= -14:
        return "작은 하락"
    return "큰 하락"


def classify_average(score: int, average: int) -> str:
    gap = score - average
    if gap >= 3:
        return "평균 위"
    if gap <= -3:
        return "평균 아래"
    return "평균 근처"


def collect_traits(row) -> list[str]:
    """v11에서 오해가 잦았던 성격을 태그로 남긴다."""
    traits = []
    score = int(row["댓글신호"])
    change = int(row["전일대비"])
    average = int(row["직전5일평균"])

    if row["수급방향"] == "BUY" and score < 40:
        traits.append("매수우위인데 신호낮음")
    if row["수급방향"] == "SELL" and score >= 60:
        traits.append("매도우위인데 신호높음")
    if score >= 60 and change <= -15:
        traits.append("높은점수인데 전일급락")
    if score < 40 and change >= 15:
        traits.append("낮은점수인데 전일급등")
    if abs(score - average) >= 20:
        traits.append("평균과 큰 차이")
    if abs(change) >= 30:
        traits.append("변화량 매우 큼")

    return traits


HISTORY_COLUMNS = ("전일신호", "전일대비", "직전5일평균")
INT_COLUMNS = ("댓글신호", "댓글수", *HISTORY_COLUMNS)


def filter_usable_rows(frame: pd.DataFrame) -> pd.DataFrame:
    """fixture로 쓸 수 있는 행만 남긴다.

    제외 대상은 두 가지다.
    - 신호상태가 ready가 아닌 행. 점수 자체가 확정되지 않았다.
    - 신호 이력(전일신호/전일대비/직전5일평균)이 비어 있는 행.
      직전 거래일 이력이 없어 교차 관계 프롬프트를 만들 수 없다.

    실험은 관계 서술 문체를 보려는 것이므로 이력이 없는 행은
    애초에 비교 대상이 되지 못한다.
    """
    total = len(frame)
    usable = frame.copy()

    if "신호상태" in usable.columns:
        not_ready = int((usable["신호상태"] != "ready").sum())
        usable = usable[usable["신호상태"] == "ready"]
    else:
        not_ready = 0

    missing = usable[list(HISTORY_COLUMNS)].isna().any(axis=1)
    no_history = int(missing.sum())
    usable = usable[~missing].copy()

    for column in INT_COLUMNS:
        usable[column] = usable[column].astype(float).round().astype(int)

    print(
        f"원본 {total}건 → 사용 가능 {len(usable)}건 "
        f"(신호상태 미준비 {not_ready}건, 신호 이력 없음 {no_history}건 제외)"
    )
    return usable.reset_index(drop=True)


def select_experiment_cases(source: pd.DataFrame, n_cases: int, seed: int):
    """
    커버리지 쿼터를 먼저 채우고 남은 자리를 방향 균형으로 채운다.

    결정적으로 동작하도록 seed 기반 고정 순서를 사용한다.
    """
    frame = source.copy()
    frame["변화구분"] = frame["전일대비"].astype(int).map(classify_change)
    frame["평균관계"] = [
        classify_average(int(s), int(a))
        for s, a in zip(frame["댓글신호"], frame["직전5일평균"])
    ]
    frame["특이성격"] = [
        ", ".join(collect_traits(row)) for _, row in frame.iterrows()
    ]

    # 특이 성격이 있는 행을 앞에 두되 순서 자체는 seed로 고정한다.
    frame["_shuffle"] = [
        abs(hash((seed, str(value)))) % 10_000_000
        for value in frame.get("보고서ID", frame.index)
    ]
    frame["_trait_count"] = frame["특이성격"].str.count(",").where(
        frame["특이성격"].str.len() > 0, -1
    )
    ordered = frame.sort_values(
        ["_trait_count", "_shuffle"], ascending=[False, True]
    )

    chosen_index = []

    def take(mask, limit=1):
        for index in ordered[mask.reindex(ordered.index, fill_value=False)].index:
            if index in chosen_index or len(chosen_index) >= n_cases:
                continue
            chosen_index.append(index)
            limit -= 1
            if limit <= 0:
                return

    # 1) 문제 성격 최소 1건씩
    for trait in (
        "매수우위인데 신호낮음",
        "매도우위인데 신호높음",
        "높은점수인데 전일급락",
        "낮은점수인데 전일급등",
        "평균과 큰 차이",
        "변화량 매우 큼",
    ):
        take(ordered["특이성격"].str.contains(trait, regex=False))

    # 2) 신호 등급 5종 최소 2건씩
    for level in LEVELS:
        take(ordered["신호강도"] == level, limit=2)

    # 3) 전일 변화 5구간 최소 1건씩
    for bucket in CHANGE_BUCKETS:
        take(ordered["변화구분"] == bucket)

    # 4) 평균 관계 3구간 최소 1건씩
    for bucket in AVERAGE_BUCKETS:
        take(ordered["평균관계"] == bucket)

    # 5) 남은 자리는 BUY/SELL 균형을 보며 채운다
    while len(chosen_index) < n_cases:
        picked = ordered.loc[chosen_index] if chosen_index else ordered.iloc[:0]
        buy_count = int((picked["수급방향"] == "BUY").sum())
        sell_count = int((picked["수급방향"] == "SELL").sum())
        wanted = "BUY" if buy_count <= sell_count else "SELL"
        before = len(chosen_index)
        take(ordered["수급방향"] == wanted)

        if len(chosen_index) == before:
            take(ordered["수급방향"].notna())

        if len(chosen_index) == before:
            break

    selected = ordered.loc[chosen_index].copy()
    selected = selected.sort_values(["기준일", "종목명"]).reset_index(drop=True)
    selected = selected.drop(columns=["_shuffle", "_trait_count"])
    return selected


FIXTURE_COLUMNS = [
    "case_id",
    "기준일",
    "종목코드",
    "종목명",
    "수급방향",
    "실제수급지수",
    "댓글신호",
    "신호강도",
    "전일신호",
    "전일대비",
    "직전5일평균",
    "댓글수",
    "변화구분",
    "평균관계",
    "특이성격",
]

if CASES_CSV.exists():
    cases_df = pd.read_csv(CASES_CSV)
    print(f"기존 fixture 로드: {len(cases_df)}건 ({CASES_CSV.name})")
else:
    if not SOURCE_REVIEW_CSV.exists():
        found = (
            sorted(item.name for item in REVIEW_DIR.glob("*.csv"))
            if REVIEW_DIR.exists()
            else []
        )
        raise FileNotFoundError(
            f"{SOURCE_REVIEW_CSV.name}이(가) 없습니다.\n"
            f"  찾은 경로: {SOURCE_REVIEW_CSV}\n"
            f"  해당 폴더의 CSV: {found or '(폴더 자체가 없음)'}\n"
            "  05_llm_report 노트북의 CSV 저장 셀까지 실행해 주세요."
        )

    source_df = filter_usable_rows(pd.read_csv(SOURCE_REVIEW_CSV))
    if len(source_df) < N_CASES:
        raise ValueError(
            f"사용 가능한 행이 {len(source_df)}건이라 요청한 {N_CASES}건을 "
            "채울 수 없습니다. N_CASES를 줄이거나 검수 CSV를 더 쌓아 주세요."
        )

    cases_df = select_experiment_cases(source_df, N_CASES, FIXTURE_SEED)
    cases_df["case_id"] = (
        cases_df["보고서ID"]
        if "보고서ID" in cases_df.columns
        else range(1, len(cases_df) + 1)
    )
    cases_df = cases_df[
        [column for column in FIXTURE_COLUMNS if column in cases_df.columns]
    ]
    cases_df.to_csv(CASES_CSV, index=False, encoding="utf-8-sig")
    print(f"fixture 신규 생성: {len(cases_df)}건 → {CASES_CSV.name}")

cases_df


In [ ]:
# Cell 5 — fixture 분포 확인
print("[수급 방향]")
print(cases_df["수급방향"].value_counts().to_string())

print("\n[신호 등급]")
print(
    cases_df["신호강도"]
    .value_counts()
    .reindex(LEVELS)
    .fillna(0)
    .astype(int)
    .to_string()
)

print("\n[전일 대비]")
print(
    cases_df["변화구분"]
    .value_counts()
    .reindex(CHANGE_BUCKETS)
    .fillna(0)
    .astype(int)
    .to_string()
)

print("\n[최근 5일 평균 관계]")
print(
    cases_df["평균관계"]
    .value_counts()
    .reindex(AVERAGE_BUCKETS)
    .fillna(0)
    .astype(int)
    .to_string()
)

print("\n[의도적으로 포함한 성격]")
traits = (
    cases_df["특이성격"].fillna("").str.split(", ").explode().str.strip()
)
print(traits[traits != ""].value_counts().to_string())


## Cell 6 — production request 변환 adapter

fixture 한 행을 production `ReportGenerationRequest`로 바꾼다.
`classify_supply_state`는 production 함수를 그대로 쓴다.

프롬프트와 검증에 쓰이지 않는 식별자(`daily_document_id`, 결과 ID,
`stock_id` 등)는 DB를 조회하지 않으므로 고정 placeholder를 넣는다.
이 값들은 저장·중복 판정에만 쓰이며 이번 실험에는 영향이 없다.


In [ ]:
# Cell 6 — fixture row → production request
PLACEHOLDER_IDS = {
    "daily_document_id": 1,
    "positive_result_id": 1,
    "negative_result_id": 2,
    "stock_id": 1,
}
PLACEHOLDER_MODEL = {
    "model_name": "ridge_supply",
    "model_version": 4,
    "artifact_schema_version": 2,
    "calibration_schema_version": 1,
}


def build_request_from_case(case: pd.Series) -> ReportGenerationRequest:
    """fixture 값을 그대로 사용한다. 신호를 다시 계산하지 않는다."""
    score = int(case["댓글신호"])
    previous = int(case["전일신호"])
    change = int(case["전일대비"])
    supply_index = float(case["실제수급지수"])
    evidence = LlmSignalEvidence(
        actual_supply_index=supply_index,
        supply_direction=case["수급방향"],
        signal_status="ready",
        comment_signal_score=score,
        signal_level=case["신호강도"],
        comment_count=int(case["댓글수"]),
        previous_signal_score=previous,
        signal_change=change,
        signal_ma5=int(case["직전5일평균"]),
    )
    return ReportGenerationRequest(
        stock_code=str(case["종목코드"]).zfill(6),
        stock_name=case["종목명"],
        model_date=date_type.fromisoformat(str(case["기준일"])),
        comment_count=int(case["댓글수"]),
        supply_state=classify_supply_state(supply_index),
        active_model_variant=(
            "positive" if case["수급방향"] == "BUY" else "negative"
        ),
        predicted_score=None,
        recognized_feature_count=None,
        evidence=evidence,
        provider=settings.provider,
        model=settings.model,
        **PLACEHOLDER_IDS,
        **PLACEHOLDER_MODEL,
    )


requests_by_case = {
    int(case["case_id"]): build_request_from_case(case)
    for _, case in cases_df.iterrows()
}
print(f"request 변환 완료: {len(requests_by_case)}건")


## Cell 7 — 프롬프트 variant 정의

프롬프트 문자열만 다르고 출력 계약(`market_commentary`, `conclusion`),
response format, temperature, model, fixture는 모두 같다.

| variant | 성격 |
|---|---|
| `A_baseline` | 현재 production 메시지 그대로 |
| `B_friendly_interpreter` | 일반 투자자용 데이터 해설자 |
| `C_mobile_briefing` | 모바일 앱 브리핑 에디터. 더 짧게 |
| `D_free_numeric` | 페르소나 최소. 숫자만 주고 자유롭게 |

D는 `전일보다 높음` 같은 자연어 힌트를 빼고 raw 숫자만 준다. LLM이 숫자
관계를 스스로 얼마나 정확히 읽는지 보기 위해서다.


In [ ]:
# Cell 7 — variant 정의
COMMON_FACT_RULES = (
    "입력에 없는 사실, 원인, 미래 전망을 만들지 마세요. "
    "주가 방향이나 투자 권유로 확대하지 마세요. "
    "숫자는 입력에 있는 값만 그대로 쓰고 새로 계산하지 마세요. "
    "signal_level 등급은 주어진 값을 바꾸지 마세요."
)
OUTPUT_RULES = (
    "market_commentary와 conclusion 두 키만 가진 JSON 객체 하나만 "
    "반환하세요. 마크다운, <think>, 부가 설명을 넣지 마세요."
)


def numeric_facts(request: ReportGenerationRequest) -> dict:
    """모든 variant가 공유하는 raw 숫자다."""
    evidence = request.evidence
    return {
        "stock_name": request.stock_name,
        "model_date": request.model_date.isoformat(),
        "individual_direction": evidence.supply_direction,
        "signal_score": evidence.comment_signal_score,
        "signal_level": evidence.signal_level,
        "previous_score": evidence.previous_signal_score,
        "signal_change": evidence.signal_change,
        "five_day_average": evidence.signal_ma5,
        "comment_count": evidence.comment_count,
    }


def relation_hints(request: ReportGenerationRequest) -> list[str]:
    """B·C에만 주는 한국어 관계 힌트다. D에는 주지 않는다."""
    evidence = request.evidence
    hints = []
    change = evidence.signal_change
    score = evidence.comment_signal_score
    average = evidence.signal_ma5

    if change is not None:
        if change > 0:
            hints.append(f"직전 거래일보다 {change}포인트 높습니다.")
        elif change < 0:
            hints.append(f"직전 거래일보다 {abs(change)}포인트 낮습니다.")
        else:
            hints.append("직전 거래일과 같습니다.")

    if score is not None and average is not None:
        if score > average:
            hints.append(f"최근 5거래일 평균 {average}점보다 높습니다.")
        elif score < average:
            hints.append(f"최근 5거래일 평균 {average}점보다 낮습니다.")
        else:
            hints.append("최근 5거래일 평균과 같습니다.")

    return hints


SIGNAL_MEANING = (
    "댓글 수급 신호는 0~100 점수입니다. 같은 수급 방향의 과거 사례와 "
    "비교한 상대 강도이며 감성 점수나 주가 예측이 아닙니다. "
    "수급 방향은 실제 개인투자자 체결량으로 계산한 별개의 관측값입니다."
)

FRIENDLY_SYSTEM = (
    "당신은 일반 투자자가 숫자를 쉽게 이해하도록 돕는 데이터 해설자입니다. "
    f"{SIGNAL_MEANING} "
    "금융 전문 용어를 굳이 쓰지 마세요. '상회', '하회', '웃돈다' 같은 "
    "표현 대신 '보다 높습니다', '최근 며칠과 비교해도 높은 편입니다'처럼 "
    "풀어 쓰세요. 'BUY'는 '개인투자자의 매수가 매도보다 많았습니다'처럼 "
    "설명해도 됩니다. 숫자 자체는 바꾸지 말고 그 숫자가 뜻하는 현재 "
    "상태와 최근 변화를 쉽게 전달하세요. "
    f"{COMMON_FACT_RULES} {OUTPUT_RULES}"
)

MOBILE_SYSTEM = (
    "당신은 모바일 투자앱에서 사용자가 빠르게 읽는 데이터 브리핑을 쓰는 "
    f"에디터입니다. {SIGNAL_MEANING} "
    "market_commentary는 2~3문장, conclusion은 1문장으로 짧게 쓰세요. "
    "같은 말을 반복하지 말고 핵심 숫자는 유지하세요. 보고서체를 피하고 "
    "문장 순서는 자유롭게 정하세요. "
    f"{COMMON_FACT_RULES} {OUTPUT_RULES}"
)

FREE_SYSTEM = (
    "제공된 사실만 사용하여 일반 사용자가 읽기 좋은 짧은 시장 브리핑을 "
    f"작성하세요. {SIGNAL_MEANING} "
    "입력에 없는 사실을 만들지 마세요. 주가 방향이나 향후 전망을 추측하지 "
    "마세요. 투자 권유를 하지 마세요. signal_level 등급은 그대로 쓰세요. "
    "나머지 문장 구성과 설명 순서는 자유롭게 결정하세요. "
    f"{OUTPUT_RULES}"
)


def _facts_block(request: ReportGenerationRequest) -> str:
    return json.dumps(
        numeric_facts(request), ensure_ascii=False, sort_keys=True
    )


def build_friendly_messages(request):
    hints = "\n".join(f"- {hint}" for hint in relation_hints(request))
    return [
        {"role": "system", "content": FRIENDLY_SYSTEM},
        {
            "role": "user",
            "content": (
                f"{request.stock_name}의 오늘 댓글 수급 신호를 설명하세요.\n"
                f"FACTS={_facts_block(request)}\n"
                f"코드가 판정한 관계입니다. 의미를 바꾸지 마세요.\n{hints}\n"
                "2~4문장으로 작성하세요."
            ),
        },
    ]


def build_mobile_messages(request):
    hints = "\n".join(f"- {hint}" for hint in relation_hints(request))
    return [
        {"role": "system", "content": MOBILE_SYSTEM},
        {
            "role": "user",
            "content": (
                f"{request.stock_name} 브리핑을 작성하세요.\n"
                f"FACTS={_facts_block(request)}\n"
                f"코드가 판정한 관계입니다. 의미를 바꾸지 마세요.\n{hints}"
            ),
        },
    ]


def build_free_messages(request):
    # 자연어 관계 힌트를 주지 않는다. 숫자 관계를 직접 읽게 한다.
    return [
        {"role": "system", "content": FREE_SYSTEM},
        {
            "role": "user",
            "content": f"FACTS={_facts_block(request)}",
        },
    ]


@dataclass(frozen=True)
class PromptVariant:
    experiment_id: str
    description: str
    build: Callable[[ReportGenerationRequest], list[dict]]


EXPERIMENTS = {
    BASELINE: PromptVariant(
        BASELINE,
        f"production {PROMPT_VERSION} 메시지 그대로",
        lambda request: list(build_report_messages(request)),
    ),
    FRIENDLY: PromptVariant(
        FRIENDLY, "일반 투자자용 데이터 해설자", build_friendly_messages
    ),
    MOBILE: PromptVariant(
        MOBILE, "모바일 앱 브리핑 에디터", build_mobile_messages
    ),
    FREE: PromptVariant(
        FREE, "페르소나 최소, 숫자만 제공", build_free_messages
    ),
}

for variant in EXPERIMENTS.values():
    print(f"{variant.experiment_id:24} {variant.description}")


In [ ]:
# Cell 8 — 공용 메시지 빌더
def build_experiment_messages(
    request: ReportGenerationRequest,
    experiment_id: str,
) -> list[dict]:
    """variant 하나를 골라 메시지를 만든다. A는 production 함수를 호출한다."""
    if experiment_id not in EXPERIMENTS:
        raise KeyError(f"정의되지 않은 variant입니다: {experiment_id}")

    return EXPERIMENTS[experiment_id].build(request)


## Cell 9 — 단일 케이스 dry-run

API를 호출하지 않고 메시지 구조만 확인한다.


In [ ]:
# Cell 9 — dry-run
DRY_RUN_CASE_ID = int(cases_df.iloc[0]["case_id"])
DRY_RUN_VARIANT = FREE

dry_request = requests_by_case[DRY_RUN_CASE_ID]
dry_messages = build_experiment_messages(dry_request, DRY_RUN_VARIANT)
dry_case = cases_df[cases_df["case_id"] == DRY_RUN_CASE_ID].iloc[0]

print(f"[CASE {DRY_RUN_CASE_ID}] {dry_case['기준일']} / {dry_case['종목명']}")
print(
    f"  수급={dry_case['수급방향']} 신호={dry_case['댓글신호']}"
    f"/{dry_case['신호강도']} 전일={dry_case['전일신호']}"
    f" 변화={dry_case['전일대비']:+d} 5일평균={dry_case['직전5일평균']}"
)
print(f"\nvariant={DRY_RUN_VARIANT}")

for message in dry_messages:
    body = message["content"]
    size = len(body.encode("utf-8"))
    print(f"\n--- {message['role']} ({size} bytes) ---")
    print(body)

total_bytes = sum(
    len(message["content"].encode("utf-8")) for message in dry_messages
)
print(f"\n전체 메시지 크기: {total_bytes} bytes")


## Cell 10 — 단일 케이스 실제 호출

`ENABLE_LLM_CALL = True`일 때만 호출한다. 배치 전에 한 건으로 먼저
확인한다.


In [ ]:
# Cell 10 — 단일 호출
def run_single(
    case_id: int,
    experiment_id: str,
    client,
) -> dict:
    """
    한 번 호출하고 결과를 그대로 기록한다.

    검증 실패로 재생성하지 않는다. 응답 원문은 어떤 경우에도 버리지
    않는다. 첫 응답 품질을 비교하는 것이 실험 목적이기 때문이다.
    """
    request = requests_by_case[case_id]
    case = cases_df[cases_df["case_id"] == case_id].iloc[0]
    messages = build_experiment_messages(request, experiment_id)
    record = {
        "case_id": case_id,
        "experiment_id": experiment_id,
        "기준일": case["기준일"],
        "종목코드": case["종목코드"],
        "종목명": case["종목명"],
        "수급방향": case["수급방향"],
        "실제수급지수": case["실제수급지수"],
        "댓글신호": case["댓글신호"],
        "신호강도": case["신호강도"],
        "전일신호": case["전일신호"],
        "전일대비": case["전일대비"],
        "직전5일평균": case["직전5일평균"],
        "댓글수": case["댓글수"],
        "특이성격": case.get("특이성격", ""),
        "system_prompt": messages[0]["content"],
        "user_prompt": messages[-1]["content"],
        "raw_response": None,
        "parse_ok": False,
        "parse_error": None,
        "market_commentary": None,
        "conclusion": None,
        "validator_ok": None,
        "rejection_reason": None,
        "provider": settings.provider,
        "model": settings.model,
        "temperature": REPORT_TEMPERATURE,
        "max_tokens": REPORT_MAX_TOKENS,
        "provider_response_id": None,
        "input_tokens": None,
        "output_tokens": None,
        "call_error": None,
    }

    try:
        completion = client.request_completion(messages)
    except (LlmReportTransportError, Exception) as error:
        record["call_error"] = f"{type(error).__name__}: {error}"
        return record

    record["raw_response"] = completion.content
    record["provider_response_id"] = completion.response_id
    record["input_tokens"] = completion.input_tokens
    record["output_tokens"] = completion.output_tokens

    if completion.refusal:
        record["parse_error"] = f"공급자 거절: {completion.refusal}"
        return record

    try:
        commentary = parse_market_commentary_response(
            content=completion.content,
            finish_reason=completion.finish_reason,
        )
    except Exception as error:
        record["parse_error"] = f"{type(error).__name__}: {error}"
        return record

    record["parse_ok"] = True
    record["market_commentary"] = commentary.market_commentary
    record["conclusion"] = commentary.conclusion

    # 검증은 관찰만 한다. 실패해도 결과를 버리거나 재생성하지 않는다.
    try:
        validate_market_commentary_response(
            request=requests_by_case[case_id],
            response=commentary,
        )
        record["validator_ok"] = True
    except Exception as error:
        record["validator_ok"] = False
        record["rejection_reason"] = str(error)

    return record


if ENABLE_LLM_CALL:
    experiment_client = OpenAICompatibleLlmReportClient(settings=settings)
    single = run_single(DRY_RUN_CASE_ID, DRY_RUN_VARIANT, experiment_client)

    print(f"raw_response:\n{single['raw_response']}\n")
    print(f"parse_ok={single['parse_ok']}  parse_error={single['parse_error']}")
    print(f"[시장코멘터리] {single['market_commentary']}")
    print(f"[한줄정리] {single['conclusion']}")
    print(f"validator_ok={single['validator_ok']}")
    print(f"사유={single['rejection_reason']}")
else:
    print("ENABLE_LLM_CALL=False 이므로 호출하지 않았습니다.")


## Cell 11 — 배치 실행

호출 전에 예상 건수를 먼저 출력한다. 한 건 실패가 전체를 중단시키지
않는다.


In [ ]:
# Cell 11 — 배치 실행
def load_existing_runs() -> pd.DataFrame:
    if RUNS_CSV.exists():
        return pd.read_csv(RUNS_CSV)

    return pd.DataFrame()


def plan_batch():
    target_ids = (
        [int(value) for value in cases_df["case_id"]]
        if RUN_CASE_IDS is None
        else [int(value) for value in RUN_CASE_IDS]
    )
    unknown = [value for value in target_ids if value not in requests_by_case]

    if unknown:
        raise KeyError(f"fixture에 없는 case_id입니다: {unknown}")

    existing = load_existing_runs()
    done = set()

    if not existing.empty and not OVERWRITE_EXISTING:
        done = {
            (int(row["case_id"]), row["experiment_id"])
            for _, row in existing.iterrows()
        }

    planned = [
        (case_id, experiment_id)
        for case_id in target_ids
        for experiment_id in RUN_VARIANTS
        if (case_id, experiment_id) not in done
    ]
    return target_ids, planned, existing


target_ids, planned_calls, existing_runs = plan_batch()

print(f"실험 케이스: {len(target_ids)}")
print(f"variant: {len(RUN_VARIANTS)} → {RUN_VARIANTS}")
print(f"예상 API 호출: {len(planned_calls)}")
print(f"이미 저장된 결과: {len(existing_runs)}건 (OVERWRITE_EXISTING={OVERWRITE_EXISTING})")
print(f"provider: {settings.provider}")
print(f"model: {settings.model}")
print(f"ENABLE_LLM_CALL: {ENABLE_LLM_CALL}")

new_records = []

if not ENABLE_LLM_CALL:
    print("\nENABLE_LLM_CALL=False 이므로 호출하지 않았습니다.")
elif not planned_calls:
    print("\n새로 호출할 조합이 없습니다.")
else:
    experiment_client = OpenAICompatibleLlmReportClient(settings=settings)

    for order, (case_id, experiment_id) in enumerate(planned_calls, start=1):
        try:
            record = run_single(case_id, experiment_id, experiment_client)
        except Exception as error:
            # 한 건 실패가 배치를 중단시키지 않는다.
            record = {
                "case_id": case_id,
                "experiment_id": experiment_id,
                "call_error": f"{type(error).__name__}: {error}",
            }

        new_records.append(record)
        status = (
            "call_error"
            if record.get("call_error")
            else ("parse_fail" if not record.get("parse_ok") else
                  ("validator_ok" if record.get("validator_ok") else "validator_fail"))
        )
        print(f"  [{order}/{len(planned_calls)}] {case_id} {experiment_id} → {status}")

print(f"\n새 결과: {len(new_records)}건")


In [ ]:
# Cell 12 — experiment_runs.csv 저장
RUN_KEY = ["case_id", "experiment_id"]

if new_records:
    new_df = pd.DataFrame(new_records)
    merged = (
        pd.concat([existing_runs, new_df], ignore_index=True)
        if not existing_runs.empty
        else new_df
    )
    # 나중 결과가 이깁니다. OVERWRITE_EXISTING=False면 애초에 재호출을
    # 하지 않으므로 기존 값이 유지됩니다.
    merged = merged.drop_duplicates(subset=RUN_KEY, keep="last")
    merged = merged.sort_values(RUN_KEY).reset_index(drop=True)
    merged.to_csv(RUNS_CSV, index=False, encoding="utf-8-sig")
    print(f"저장: {RUNS_CSV.name} ({len(merged)}행)")
    runs_df = merged
else:
    runs_df = existing_runs
    print(f"새 결과가 없어 기존 파일을 사용합니다: {len(runs_df)}행")

runs_df.head()


## Cell 13 — variant별 성공·거부 요약

`validator_ok`는 규칙 통과 여부일 뿐 보고서 품질이 아니다. 거부가 많은
variant가 반드시 나쁜 것은 아니며, 그 판단은 Cell 14와 수동 평가로 한다.


In [ ]:
# Cell 13 — 요약
if runs_df.empty:
    print("집계할 실행 결과가 없습니다.")
    summary_df = pd.DataFrame()
else:
    grouped = runs_df.groupby("experiment_id")
    summary_df = pd.DataFrame(
        {
            "호출": grouped.size(),
            "호출오류": grouped["call_error"].apply(lambda s: s.notna().sum()),
            "parse성공": grouped["parse_ok"].apply(lambda s: (s == True).sum()),
            "parse실패": grouped["parse_ok"].apply(lambda s: (s != True).sum()),
            "validator통과": grouped["validator_ok"].apply(
                lambda s: (s == True).sum()
            ),
            "validator거부": grouped["validator_ok"].apply(
                lambda s: (s == False).sum()
            ),
        }
    ).reset_index()

    print("[거부 사유별]")
    reasons = runs_df.loc[
        runs_df["validator_ok"] == False, "rejection_reason"
    ].dropna()

    if reasons.empty:
        print("  없음")
    else:
        print(
            reasons.str.split(":").str[0].value_counts().to_string()
        )

summary_df


## Cell 14 — 보고서 원문 비교

**이 셀이 검수의 중심이다.** 요약하지 않고 LLM이 생성한 문장을 그대로
출력한다.


In [ ]:
# Cell 14 — 원문 비교 출력
def print_case_reports(case_id: int) -> None:
    case = cases_df[cases_df["case_id"] == case_id].iloc[0]
    rows = runs_df[runs_df["case_id"] == case_id]

    print("=" * 60)
    print(f"[CASE {case_id}]")
    print(f"{case['기준일']} / {case['종목명']}")
    print("입력")
    print(f"- 수급: {case['수급방향']}")
    print(f"- 신호: {case['댓글신호']} / {case['신호강도']}")
    print(f"- 전일: {case['전일신호']}")
    print(f"- 변화: {int(case['전일대비']):+d}")
    print(f"- 최근 5일 평균: {case['직전5일평균']}")

    if case.get("특이성격"):
        print(f"- 성격: {case['특이성격']}")

    for experiment_id in RUN_VARIANTS:
        matched = rows[rows["experiment_id"] == experiment_id]
        print()
        print(f"[{experiment_id}]")

        if matched.empty:
            print("(결과 없음)")
            print("-" * 60)
            continue

        row = matched.iloc[0]

        if pd.notna(row.get("call_error")):
            print(f"호출 오류: {row['call_error']}")
            print("-" * 60)
            continue

        if row.get("parse_ok") != True:
            print(f"파싱 실패: {row.get('parse_error')}")
            print(f"raw_response: {row.get('raw_response')}")
            print("-" * 60)
            continue

        print("[시장코멘터리]")
        print(row["market_commentary"])
        print("[한줄정리]")
        print(row["conclusion"])
        print(f"Validator: {'PASS' if row['validator_ok'] == True else 'FAIL'}")

        if row["validator_ok"] != True:
            print(f"사유: {row['rejection_reason']}")

        print("-" * 60)

    print("=" * 60)
    print()


if runs_df.empty:
    print("출력할 결과가 없습니다.")
else:
    for case_id in sorted(runs_df["case_id"].unique()):
        print_case_reports(int(case_id))


In [ ]:
# Cell 15 — wide 비교표
if runs_df.empty:
    print("비교표를 만들 결과가 없습니다.")
    comparison_df = pd.DataFrame()
else:
    base_columns = [
        "case_id",
        "기준일",
        "종목명",
        "수급방향",
        "댓글신호",
        "신호강도",
        "전일신호",
        "전일대비",
        "직전5일평균",
        "특이성격",
    ]
    comparison_df = (
        cases_df[[c for c in base_columns if c in cases_df.columns]]
        .copy()
        .set_index("case_id")
    )

    for experiment_id in EXPERIMENTS:
        prefix = experiment_id.split("_")[0]
        subset = (
            runs_df[runs_df["experiment_id"] == experiment_id]
            .set_index("case_id")
        )

        for source, suffix in (
            ("market_commentary", "market_commentary"),
            ("conclusion", "conclusion"),
            ("validator_ok", "validator_ok"),
            ("rejection_reason", "rejection_reason"),
        ):
            if source in subset.columns:
                comparison_df[f"{prefix}_{suffix}"] = subset[source]

    comparison_df = comparison_df.reset_index()
    comparison_df.to_csv(COMPARISON_CSV, index=False, encoding="utf-8-sig")
    print(f"저장: {COMPARISON_CSV.name} ({len(comparison_df)}행)")

comparison_df.head()


## Cell 16 — 수동 평가표

사람이 `좋음 / 보통 / 나쁨`을 직접 적는 파일이다. 이미 적어둔 평가는
덮어쓰지 않고 새 결과만 추가한다.


In [ ]:
# Cell 16 — manual_review.csv
REVIEW_INPUT_COLUMNS = [
    "case_id",
    "experiment_id",
    "기준일",
    "종목명",
    "수급방향",
    "댓글신호",
    "신호강도",
    "전일대비",
    "직전5일평균",
    "market_commentary",
    "conclusion",
    "validator_ok",
    "rejection_reason",
]
SCORE_COLUMNS = [
    "사실정확성",
    "개념정확성",
    "자연스러움",
    "정보성",
    "서비스적합성",
    "메모",
]

if runs_df.empty:
    print("평가표를 만들 결과가 없습니다.")
    manual_review_df = pd.DataFrame()
else:
    fresh = runs_df[
        [c for c in REVIEW_INPUT_COLUMNS if c in runs_df.columns]
    ].copy()

    for column in SCORE_COLUMNS:
        fresh[column] = ""

    if MANUAL_REVIEW_CSV.exists():
        saved = pd.read_csv(MANUAL_REVIEW_CSV)
        saved_keys = set(
            zip(saved["case_id"].astype(int), saved["experiment_id"])
        )
        added = fresh[
            ~fresh.apply(
                lambda row: (int(row["case_id"]), row["experiment_id"])
                in saved_keys,
                axis=1,
            )
        ]
        manual_review_df = pd.concat([saved, added], ignore_index=True)
        print(f"기존 평가 {len(saved)}행 유지, 신규 {len(added)}행 추가")
    else:
        manual_review_df = fresh
        print(f"평가표 신규 생성: {len(fresh)}행")

    manual_review_df = manual_review_df.sort_values(
        ["case_id", "experiment_id"]
    ).reset_index(drop=True)
    manual_review_df.to_csv(
        MANUAL_REVIEW_CSV, index=False, encoding="utf-8-sig"
    )
    print(f"저장: {MANUAL_REVIEW_CSV.name}")
    print("평가는 좋음 / 보통 / 나쁨 세 단계로 적습니다.")

manual_review_df.head()
